In [4]:
# -------------------------
# Reload saved models + evaluate
# -------------------------
import os, joblib, numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Helper metrics
def metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "RMSE": float(np.sqrt(mse)),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }

# 1) Attempt to load models (many possible file names we used)
model_xgb = None
gbm = None
meta = None
model_lstm = None

# XGBoost: could be saved as xgboost_week6.model (Booster/UBJSON) or as xgb_best_week7.pkl (sklearn wrapper)
try:
    import xgboost as xgb
    if os.path.exists("xgboost_week6.model"):
        try:
            # Try loading as Booster
            model_xgb = xgb.Booster()
            model_xgb.load_model("xgboost_week6.model")
            print("Loaded XGBoost Booster from xgboost_week6.model")
            xgb_is_booster = True
        except Exception as e:
            print("Found xgboost_week6.model but failed to load as Booster:", e)
            model_xgb = None
    elif os.path.exists("xgb_best_week7.pkl"):
        model_xgb = joblib.load("xgb_best_week7.pkl")
        print("Loaded XGBoost sklearn wrapper from xgb_best_week7.pkl")
        xgb_is_booster = False
    else:
        # try sklearn wrapper common name
        if os.path.exists("xgb_best_week7.joblib"):
            model_xgb = joblib.load("xgb_best_week7.joblib"); xgb_is_booster = False; print("Loaded XGBoost sklearn wrapper")
except Exception as e:
    print("xgboost import/load error:", e)

# LightGBM: may be Booster saved as lightgbm_week6.txt or sklearn wrapper joblib
try:
    import lightgbm as lgb
    if os.path.exists("lightgbm_week6.txt"):
        try:
            gbm = lgb.Booster(model_file="lightgbm_week6.txt")
            print("Loaded LightGBM Booster from lightgbm_week6.txt")
        except Exception as e:
            print("Found lightgbm_week6.txt but failed to load as Booster:", e); gbm = None
    elif os.path.exists("lgb_best_week7.pkl"):
        gbm = joblib.load("lgb_best_week7.pkl"); print("Loaded LightGBM sklearn wrapper from lgb_best_week7.pkl")
    elif os.path.exists("lightgbm_week6.joblib"):
        gbm = joblib.load("lightgbm_week6.joblib"); print("Loaded LightGBM sklearn wrapper")
except Exception as e:
    print("lightgbm import/load error:", e)

# Meta model (Ridge) - we saved as week6_meta_ridge.pkl or week6_meta_ridge.joblib
for meta_fname in ["week6_meta_ridge.pkl", "week6_meta_ridge.joblib", "week6_meta_ridge.pkl"]:
    if os.path.exists(meta_fname):
        try:
            meta = joblib.load(meta_fname)
            print("Loaded meta model from", meta_fname)
            break
        except Exception as e:
            print("Failed loading meta model", meta_fname, ":", e)

# LSTM model - check common save locations
try:
    from tensorflow.keras.models import load_model
    for p in ["models/encoder_decoder_lstm.h5", "week5_outputs/encoder_decoder_fast.h5", "best_lstm_week7.h5", "best_lstm_week7"]:
        if os.path.exists(p):
            try:
                model_lstm = load_model(p)
                print("Loaded LSTM model from", p)
                break
            except Exception as e:
                print("Found LSTM file", p, "but load failed:", e)
except Exception as e:
    print("Keras load error:", e)

# 2) Prepare X_test / y_test if not in memory
X_test = globals().get('X_test', None)
y_test = globals().get('y_test', None)
feature_cols = None

if X_test is None or y_test is None:
    # Try to reconstruct from saved feature list + original CSV
    print("X_test or y_test not found in memory — attempting to rebuild from files.")
    if os.path.exists("week6_feature_cols.pkl") and os.path.exists("player_features_final.csv"):
        try:
            feature_cols = joblib.load("week6_feature_cols.pkl")
            full_df = pd.read_csv("player_features_final.csv")
            # run same feature prep as Week6 (simple conservative version)
            # ensure required numeric columns exist
            for c in ['market_value_in_eur','highest_market_value_in_eur','sentiment_score']:
                if c not in full_df.columns:
                    full_df[c] = 0.0
            # simple label encoding fallback for small cardinality columns:
            for c in ['position','current_club_name','country_of_citizenship']:
                if c in full_df.columns and c+'_le' not in full_df.columns:
                    full_df[c+'_le'] = full_df[c].astype(str).factorize()[0]
            # construct X using feature_cols
            X_all = full_df.reindex(columns=feature_cols).fillna(0)
            y_all = pd.to_numeric(full_df['market_value_in_eur'], errors='coerce').fillna(0)
            # split same way as earlier using train/test sizes if available; fallback to simple split
            from sklearn.model_selection import train_test_split
            X_train_tmp, X_test, y_train_tmp, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=42)
            print("Rebuilt X_test and y_test from player_features_final.csv using feature list.")
        except Exception as e:
            print("Failed to rebuild X_test from files:", e)
    else:
        # fallback: check week6_preds_comparison.csv for actual & prediction columns
        if os.path.exists("week6_preds_comparison.csv"):
            dfc = pd.read_csv("week6_preds_comparison.csv")
            if 'actual' in dfc.columns:
                y_test = dfc['actual'].values
                # create X_test as zeros placeholder (we only need y_test to compute metrics for saved preds)
                X_test = None
                print("Loaded actuals from week6_preds_comparison.csv")
        else:
            print("No files found to rebuild test set. Please ensure player_features_final.csv and week6_feature_cols.pkl exist.")
else:
    print("X_test and y_test found in memory; using them.")

# 3) Evaluate all loaded models
results = {}

# Evaluate XGBoost
if model_xgb is not None:
    try:
        # If model_xgb is Booster, use DMatrix
        if isinstance(model_xgb, (xgb.core.Booster,)) and X_test is not None:
            dtest = xgb.DMatrix(X_test)
            preds_xgb = model_xgb.predict(dtest)
        else:
            # sklearn wrapper
            preds_xgb = model_xgb.predict(X_test)
        results['XGBoost'] = metrics(y_test, preds_xgb)
        print("XGBoost evaluated.")
    except Exception as e:
        print("XGBoost evaluation failed:", e)

# Evaluate LightGBM
if gbm is not None:
    try:
        if isinstance(gbm, lgb.Booster):
            preds_lgb = gbm.predict(X_test)
        else:
            preds_lgb = gbm.predict(X_test)
        results['LightGBM'] = metrics(y_test, preds_lgb)
        print("LightGBM evaluated.")
    except Exception as e:
        print("LightGBM evaluation failed:", e)

# Evaluate meta model (stacked)
if meta is not None:
    try:
        # Need base model predictions to form meta features; attempt to use preds_xgb/preds_lgb
        meta_feats = []
        if 'XGBoost' in results:
            meta_feats.append(preds_xgb)
        else:
            meta_feats.append(np.zeros(len(y_test)))
        if 'LightGBM' in results:
            meta_feats.append(preds_lgb)
        else:
            meta_feats.append(np.zeros(len(y_test)))
        # add LSTM pred if available as feature in test_df_f
        if 'test_df_f' in globals() and 'lstm_pred_next' in globals().get('test_df_f').columns:
            meta_feats.append(test_df_f['lstm_pred_next'].values)
        else:
            meta_feats.append(np.zeros(len(y_test)))
        meta_X = np.vstack(meta_feats).T
        preds_meta = meta.predict(meta_X)
        results['Stacked Ensemble'] = metrics(y_test, preds_meta)
        print("Meta model evaluated.")
    except Exception as e:
        print("Meta evaluation failed:", e)

# Evaluate LSTM if predictions exist (y_pred or saved csv)
if 'y_pred' in globals() and 'y_true' in globals():
    try:
        results['LSTM'] = metrics(y_true.flatten(), y_pred.flatten())
        print("LSTM evaluated from memory variables.")
    except Exception as e:
        print("LSTM evaluation failed:", e)
else:
    # try loading saved encoder-decoder predictions CSV
    if os.path.exists("encoder_decoder_predictions.csv") and os.path.exists("encoder_decoder_true.csv"):
        try:
            yp = pd.read_csv("encoder_decoder_predictions.csv").values
            yt = pd.read_csv("encoder_decoder_true.csv").values
            # evaluate for horizon 1 using column 0
            results['LSTM'] = metrics(yt[:,0], yp[:,0])
            print("LSTM evaluated from saved CSVs (horizon t+1).")
        except Exception as e:
            print("Failed evaluating LSTM from saved CSVs:", e)

# 4) Print & save results table
if results:
    df_res = pd.DataFrame(results).T
    print("\nEvaluation results:")
    print(df_res)
    df_res.to_csv("week7_model_metrics.csv")
    print("\nSaved week7_model_metrics.csv")
else:
    print("No models were evaluated — results empty. Check that model files and test set exist.")


C:\Users\saura\AppData\Local\Temp\ipykernel_28944\2867027603.py:29: UserWarning: [20:44:02] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `model`. Using UBJSON (`ubj`) as a guess.
  model_xgb.load_model("xgboost_week6.model")


Loaded XGBoost Booster from xgboost_week6.model
Loaded LightGBM Booster from lightgbm_week6.txt
Loaded meta model from week6_meta_ridge.pkl
Found LSTM file models/encoder_decoder_lstm.h5 but load failed: Could not deserialize 'keras.metrics.mse' because it is not a KerasSaveable subclass
X_test or y_test not found in memory — attempting to rebuild from files.
Rebuilt X_test and y_test from player_features_final.csv using feature list.
XGBoost evaluated.
LightGBM evaluated.
Meta model evaluated.
LSTM evaluated from saved CSVs (horizon t+1).

Evaluation results:
                         RMSE         MAE        R2
XGBoost              0.229810    0.160826  0.947805
LightGBM             0.183742    0.119843  0.966634
Stacked Ensemble     0.220824    0.155154  0.951807
LSTM              1062.714925  795.774724  0.986230

Saved week7_model_metrics.csv


In [7]:
# Rebuild features safely (create missing engineered cols), then produce X_train/X_val/X_test
import os, joblib
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load main dataframe
df_path = "player_features_final.csv"
if not os.path.exists(df_path):
    raise SystemExit(f"File not found: {df_path} — place your cleaned features CSV at this path.")

df = pd.read_csv(df_path)
print("Loaded", df_path, "rows:", len(df))

# Ensure numeric market value columns exist
if 'market_value_in_eur' not in df.columns:
    alt = [c for c in ['market_value','market_value_eur','value','value_eur'] if c in df.columns]
    if alt:
        df['market_value_in_eur'] = pd.to_numeric(df[alt[0]], errors='coerce')
        print(f"Mapped {alt[0]} -> market_value_in_eur")
    else:
        df['market_value_in_eur'] = 0.0
        print("No market value column found; created market_value_in_eur filled with 0.0")

if 'highest_market_value_in_eur' not in df.columns:
    # try similar names, else duplicate current market value (safe fallback)
    alt2 = [c for c in ['highest_market_value','highest_market_value_eur'] if c in df.columns]
    if alt2:
        df['highest_market_value_in_eur'] = pd.to_numeric(df[alt2[0]], errors='coerce').fillna(df['market_value_in_eur'])
        print(f"Mapped {alt2[0]} -> highest_market_value_in_eur")
    else:
        df['highest_market_value_in_eur'] = df['market_value_in_eur']
        print("No highest_market_value_in_eur found; set equal to market_value_in_eur (fallback)")

# 1) Create value_to_peak_ratio (safe division)
df['value_to_peak_ratio'] = df['market_value_in_eur'] / df['highest_market_value_in_eur'].replace({0: np.nan})
df['value_to_peak_ratio'] = df['value_to_peak_ratio'].fillna(0.0)

# 2) Create / load lstm_pred_next
if 'lstm_pred_next' not in df.columns:
    # try to load encoder-decoder predictions
    pred_file = None
    for cand in ["encoder_decoder_predictions.csv", "encoder_decoder_preds_fast.csv", "encoder_decoder_predictions_v1.csv", "week5_outputs/encoder_decoder_predictions.csv"]:
        if os.path.exists(cand):
            pred_file = cand
            break

    if pred_file:
        try:
            preds_df = pd.read_csv(pred_file)
            # try to use first column 'pred_t+1' or first column available
            if 'pred_t+1' in preds_df.columns:
                pred_col = 'pred_t+1'
            else:
                pred_col = preds_df.columns[0]
            lstm_vals = preds_df[pred_col].values
            # align length: if different, tile or pad with last value/zeros
            if len(lstm_vals) == len(df):
                df['lstm_pred_next'] = lstm_vals
            else:
                # if preds shorter, pad with last value; if longer, truncate
                if len(lstm_vals) < len(df):
                    pad = np.full(len(df)-len(lstm_vals), lstm_vals[-1] if len(lstm_vals)>0 else 0.0)
                    df['lstm_pred_next'] = np.concatenate([lstm_vals, pad])
                else:
                    df['lstm_pred_next'] = lstm_vals[:len(df)]
            df['lstm_pred_next'] = pd.to_numeric(df['lstm_pred_next'], errors='coerce').fillna(0.0)
            print(f"Loaded LSTM predictions from {pred_file} into lstm_pred_next.")
        except Exception as e:
            print("Failed loading preds file, falling back to zeros. Error:", e)
            df['lstm_pred_next'] = 0.0
    else:
        df['lstm_pred_next'] = 0.0
        print("No LSTM predictions file found; created lstm_pred_next filled with 0.0")

# 3) Label-encode categorical columns into *_le if missing
cat_cols = {
    'position': 'position_le',
    'current_club_name': 'current_club_name_le',
    'country_of_citizenship': 'country_of_citizenship_le'
}
for src, tgt in cat_cols.items():
    if tgt not in df.columns:
        if src in df.columns:
            df[tgt] = pd.factorize(df[src].astype(str))[0]
            print(f"Created label encoded column: {tgt} from {src}")
        else:
            df[tgt] = 0
            print(f"{src} not found: created {tgt} filled with 0")

# 4) Ensure sentiment_score exists
if 'sentiment_score' not in df.columns:
    df['sentiment_score'] = 0.0
    print("Created sentiment_score filled with 0.0")

# 5) Now select final feature list (same as Week6)
feature_cols = [
    'market_value_in_eur',
    'highest_market_value_in_eur',
    'value_to_peak_ratio',
    'lstm_pred_next',
    'position_le',
    'current_club_name_le',
    'country_of_citizenship_le',
    'sentiment_score'
]

# Verify features present
missing_feats = [c for c in feature_cols if c not in df.columns]
if missing_feats:
    print("Warning — missing features (will fill with zeros):", missing_feats)
    for c in missing_feats:
        df[c] = 0.0

# Drop rows where target is missing and reset index
df = df.dropna(subset=['market_value_in_eur']).reset_index(drop=True)

# 6) Build X, y and splits
X_all = df[feature_cols].astype(float).values
y_all = df['market_value_in_eur'].astype(float).values

# Standard train/val/test splits (same proportions as Week6)
X_temp, X_test, y_temp, y_test = train_test_split(X_all, y_all, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.10, random_state=42)

# 7) Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Save scaler and feature list for later reproducibility
joblib.dump(scaler, "week7_feature_scaler.save")
joblib.dump(feature_cols, "week7_feature_cols.pkl")
print("Saved week7_feature_scaler.save and week7_feature_cols.pkl")

# Print shapes and a quick preview
print("Shapes -> X_train, X_val, X_test:", X_train.shape, X_val.shape, X_test.shape)
print("y shapes ->", y_train.shape, y_val.shape, y_test.shape)
print("Feature columns:", feature_cols)


Loaded player_features_final.csv rows: 20589
Loaded LSTM predictions from encoder_decoder_predictions.csv into lstm_pred_next.
Created label encoded column: position_le from position
Created label encoded column: current_club_name_le from current_club_name
Created label encoded column: country_of_citizenship_le from country_of_citizenship
Saved week7_feature_scaler.save and week7_feature_cols.pkl
Shapes -> X_train, X_val, X_test: (14823, 8) (1648, 8) (4118, 8)
y shapes -> (14823,) (1648,) (4118,)
Feature columns: ['market_value_in_eur', 'highest_market_value_in_eur', 'value_to_peak_ratio', 'lstm_pred_next', 'position_le', 'current_club_name_le', 'country_of_citizenship_le', 'sentiment_score']


In [6]:
# Robust prediction for Booster across xgboost versions
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# dtest is xgb.DMatrix(X_test) from previous cell
try:
    # preferred: use best_iteration / iteration_range if available
    best_iter = getattr(bst, "best_iteration", None)
    if best_iter is not None:
        # many xgboost versions accept iteration_range=(0, best_iter)
        preds_test = bst.predict(dtest, iteration_range=(0, best_iter))
    else:
        # fallback if best_iteration not present
        preds_test = bst.predict(dtest)
except TypeError:
    # some versions raise TypeError for iteration_range; fall back
    try:
        preds_test = bst.predict(dtest)
    except Exception as e:
        raise RuntimeError("Booster.predict failed with both iteration_range and plain call.") from e

# compute metrics
rmse_test = mean_squared_error(y_test, preds_test) ** 0.5
mae_test = mean_absolute_error(y_test, preds_test)
print(f"Final tuned XGBoost (booster) test RMSE: {rmse_test:.6f}, MAE: {mae_test:.6f}")


Final tuned XGBoost (booster) test RMSE: 0.003288, MAE: 0.000082


In [ ]:
# === LightGBM tuning: RandomizedSearchCV (no early stopping) + refit with lgb.train callbacks ===
import os, joblib, numpy as np, pandas as pd, warnings
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
import lightgbm as lgb
warnings.filterwarnings("ignore")

# assume X_train, X_val, X_test, y_train, y_val, y_test are in memory
# if not, run the feature-rebuild cell we used earlier to recreate them.

# 1) Randomized search on sklearn wrapper (fast)
lgb_model = lgb.LGBMRegressor(random_state=42, n_jobs=-1)

param_dist_lgb = {
    "n_estimators": [200, 400, 600, 800],
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}

search_lgb = RandomizedSearchCV(
    lgb_model,
    param_distributions=param_dist_lgb,
    n_iter=12,            # small; increase if you have time
    scoring="neg_root_mean_squared_error",
    cv=3,
    random_state=42,
    verbose=2,
    n_jobs=-1,
    refit=True
)

print("Running RandomizedSearchCV for LightGBM...")
search_lgb.fit(X_train, y_train)
print("Random search done. Best params:", search_lgb.best_params_)
best_params = search_lgb.best_params_.copy()

# 2) Refit with lgb.train to enable callbacks (early stopping)
# Build datasets for lgb.train (use original unscaled X arrays if you saved them; we have scaled ones)
# lgb.Dataset expects numpy arrays (can be scaled)
dtrain = lgb.Dataset(X_train, label=y_train)
dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

params_train = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': best_params.get('learning_rate', 0.05),
    'num_leaves': int(best_params.get('num_leaves', 31)),
    'feature_fraction': best_params.get('colsample_bytree', 0.8),
    'bagging_fraction': best_params.get('subsample', 0.8),
    'seed': 42
}
num_boost_round = int(best_params.get('n_estimators', 500))

callbacks = [
    lgb.early_stopping(stopping_rounds=30),
    lgb.log_evaluation(period=50)
]

print("Refitting LightGBM with lgb.train and callbacks...")
gbm = lgb.train(params_train, dtrain, num_boost_round=num_boost_round, valid_sets=[dtrain, dval], callbacks=callbacks)

# Evaluate on test
y_pred_lgb = gbm.predict(X_test, num_iteration=gbm.best_iteration)
rmse_lgb = mean_squared_error(y_test, y_pred_lgb) ** 0.5
mae_lgb  = mean_absolute_error(y_test, y_pred_lgb)
print(f"Tuned LightGBM test RMSE: {rmse_lgb:.6f}, MAE: {mae_lgb:.6f}")

# Save models
gbm.save_model("lgb_tuned_week7.txt")
joblib.dump(search_lgb.best_estimator_, "lgb_tuned_week7.pkl")
print("Saved lgb_tuned_week7.txt and lgb_tuned_week7.pkl")

Running RandomizedSearchCV for LightGBM...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000331 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1026
[LightGBM] [Info] Number of data points in the train set: 14823, number of used features: 7
[LightGBM] [Info] Start training from score -0.004290
Random search done. Best params: {'subsample': 0.8, 'num_leaves': 31, 'n_estimators': 800, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Refitting LightGBM with lgb.train and callbacks...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000305 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1026
[LightGBM] [Info] Number of data points in the train set: 14823, number of used features: 7
[LightGBM] [Info] Start training from score -0.004290
Training until validati

In [12]:
# Robust data-prep for LSTM tuning: builds or loads synth_df, then creates sequences.
import os, math, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# PARAMETERS (you can tune these)
ORIG_SEQ_LEN = 12
OUT_STEPS = 3
MIN_SEQ_LEN = 3        # lower bound if not enough history
SYNTH_MULT = 1         # multiplier for synthetic series length (increase if short)

# 1) Load synth_df if it exists, else synthesize short series from player_features_final.csv
synth_path = "player_synth_timeseries.csv"
if os.path.exists(synth_path):
    synth_df = pd.read_csv(synth_path)
    print(f"Loaded existing {synth_path} with {len(synth_df)} rows.")
else:
    # Try to synthesize from player_features_final.csv
    src = "player_features_final.csv"
    if not os.path.exists(src):
        raise SystemExit(f"Required file not found: {src}. Place it in working dir or provide player_synth_timeseries.csv.")
    df0 = pd.read_csv(src)
    print(f"Loaded {src} with {len(df0)} rows — synthesizing short time series per player (demo).")
    # Minimal required columns: player_id (or create), market_value_in_eur, sentiment_score (optional), injury_flag optional
    if 'player_id' not in df0.columns:
        df0 = df0.reset_index().rename(columns={'index':'player_id'})
    if 'market_value_in_eur' not in df0.columns:
        # try common alternates
        alt = [c for c in ['market_value','market_value_eur','value','value_eur'] if c in df0.columns]
        if alt:
            df0['market_value_in_eur'] = pd.to_numeric(df0[alt[0]], errors='coerce').fillna(0.0)
        else:
            df0['market_value_in_eur'] = 0.0
    if 'sentiment_score' not in df0.columns:
        df0['sentiment_score'] = 0.0
    # choose synthesis length
    SYNTH_STEPS = max(ORIG_SEQ_LEN + OUT_STEPS + 5, 10) * SYNTH_MULT
    rows = []
    for _, r in df0.iterrows():
        pid = r['player_id']
        cur = float(r.get('market_value_in_eur', 0.0) or 0.0)
        peak = max(cur * 1.2, cur + 1e5)
        vals = np.linspace(peak, cur, int(SYNTH_STEPS))
        noise = 1 + np.random.normal(0, 0.04, size=int(SYNTH_STEPS))
        vals_noisy = (vals * noise).clip(min=0)
        base_sent = float(r.get('sentiment_score', 0.0) or 0.0)
        sents = base_sent + np.random.normal(0, 0.02, size=int(SYNTH_STEPS))
        inj = np.random.binomial(1, 0.03, size=int(SYNTH_STEPS))
        for t in range(int(SYNTH_STEPS)):
            rows.append({
                'player_id': pid,
                'timestep': int(t),
                'market_value_in_eur': float(vals_noisy[t]),
                'sentiment_score': float(sents[t]),
                'injury_flag': int(inj[t])
            })
    synth_df = pd.DataFrame(rows)
    synth_df.to_csv(synth_path, index=False)
    print(f"Synthesized time-series saved to {synth_path} with {len(synth_df)} rows.")

# 2) Sequence builder function
def build_sequences_local(df_ts, feature_cols, value_col='market_value_in_eur', seq_len=ORIG_SEQ_LEN, pred_steps=OUT_STEPS, player_col='player_id', time_col='timestep'):
    X_list, y_list = [], []
    grouped = df_ts.groupby(player_col)
    for pid, g in grouped:
        if time_col in g.columns:
            g = g.sort_values(time_col)
        else:
            g = g.reset_index(drop=True)
        if len(g) < seq_len + pred_steps:
            continue
        arr = g[feature_cols].values
        vals = g[value_col].values
        for i in range(len(g) - seq_len - pred_steps + 1):
            X_list.append(arr[i:i+seq_len])
            y_list.append(vals[i+seq_len:i+seq_len+pred_steps])
    if not X_list:
        return np.empty((0,)), np.empty((0,))
    X = np.array(X_list)
    y = np.array(y_list)
    return X, y

# 3) Choose feature columns present
feat_cands = ['market_value_in_eur','sentiment_score','injury_flag']
feature_cols = [c for c in feat_cands if c in synth_df.columns]
if not feature_cols:
    raise SystemExit("No feature columns found in synth_df. Need at least 'market_value_in_eur'.")

print("Using feature columns:", feature_cols)

# 4) Try building sequences; if none found, reduce SEQ_LEN gradually until we find sequences
seq_len = ORIG_SEQ_LEN
X_ed, y_ed = build_sequences_local(synth_df, feature_cols, seq_len=seq_len, pred_steps=OUT_STEPS)
attempt = 0
while (X_ed.size == 0 or y_ed.size == 0) and seq_len >= MIN_SEQ_LEN:
    attempt += 1
    print(f"No sequences with SEQ_LEN={seq_len}. Reducing seq_len...")
    seq_len = seq_len - 1
    X_ed, y_ed = build_sequences_local(synth_df, feature_cols, seq_len=seq_len, pred_steps=OUT_STEPS)
    if attempt > (ORIG_SEQ_LEN - MIN_SEQ_LEN + 2):
        break

if X_ed.size == 0 or y_ed.size == 0:
    raise SystemExit(f"Could not build sequences with SEQ_LEN down to {seq_len}. Try synthesizing longer series or lower OUT_STEPS.")

print(f"Built sequences: X shape = {X_ed.shape}, y shape = {y_ed.shape} (using seq_len={seq_len})")

# 5) reshape y to 3d for encoder-decoder and train-test split
y_ed_3d = y_ed.reshape(y_ed.shape[0], y_ed.shape[1], 1)

Xtr, Xte, ytr, yte = train_test_split(X_ed, y_ed_3d, test_size=0.2, random_state=42)

# 6) Scale features per-feature (fit on train)
from sklearn.preprocessing import MinMaxScaler
nf = Xtr.shape[2]
X_scaler = MinMaxScaler().fit(Xtr.reshape(-1, nf))
Xtr_s = X_scaler.transform(Xtr.reshape(-1, nf)).reshape(Xtr.shape)
Xte_s = X_scaler.transform(Xte.reshape(-1, nf)).reshape(Xte.shape)

y_scaler = MinMaxScaler().fit(ytr.reshape(-1,1))
ytr_s = y_scaler.transform(ytr.reshape(-1,1)).reshape(ytr.shape)
yte_s = y_scaler.transform(yte.reshape(-1,1)).reshape(yte.shape)

# 7) expose key variables for tuning cell
# SEQ_LEN may have changed to lower value
SEQ_LEN = seq_len
OUT_STEPS = OUT_STEPS
Xtr_s, Xte_s, ytr_s, yte_s, nf = Xtr_s, Xte_s, ytr_s, yte_s, nf

print("\nReady for LSTM tuning. Variables created:")
print("SEQ_LEN:", SEQ_LEN, "OUT_STEPS:", OUT_STEPS, "nf:", nf)
print("Xtr_s shape:", Xtr_s.shape, "Xte_s shape:", Xte_s.shape)
print("ytr_s shape:", ytr_s.shape, "yte_s shape:", yte_s.shape)

import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Dense, RepeatVector, TimeDistributed, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# 1. Hyperparameter Search Space
# -----------------------------
param_grid = {
    "units": [64, 128, 256],
    "dropout": [0.1, 0.2, 0.3],
    "lr": [0.001, 0.0005],
    "batch_size": [64, 128]
}

best_rmse = float("inf")
best_params = None
best_model = None

print("🔍 Starting LSTM hyperparameter tuning...")

# -----------------------------
# 2. Tuning Loop
# -----------------------------
for units in param_grid["units"]:
    for dropout in param_grid["dropout"]:
        for lr in param_grid["lr"]:
            for batch in param_grid["batch_size"]:

                print(f"\nTesting → units={units}, dropout={dropout}, lr={lr}, batch={batch}")

                # -----------------------------
                # Build LSTM Encoder–Decoder model
                # -----------------------------
                inp = tf.keras.Input(shape=(Xtr_s.shape[1], Xtr_s.shape[2]))

                enc = LSTM(units, return_sequences=False)(inp)
                rep = RepeatVector(OUT_STEPS)(enc)
                dec = LSTM(units, return_sequences=True)(rep)
                dec = TimeDistributed(Dense(64, activation='relu'))(dec)
                out = TimeDistributed(Dense(1))(dec)

                model = Model(inputs=inp, outputs=out)
                model.compile(
                    optimizer=Adam(learning_rate=lr),
                    loss="mse"
                )

                # Train small number of epochs to keep tuning fast
                history = model.fit(
                    Xtr_s, ytr_s,
                    validation_split=0.1,
                    epochs=20,
                    batch_size=batch,
                    verbose=0
                )

                # Evaluate on validation set
                pred = model.predict(Xte_s, verbose=0)
                pred = pred.reshape(pred.shape[0], pred.shape[1])
                true = yte_s.reshape(yte_s.shape[0], yte_s.shape[1])

                rmse = np.sqrt(mean_squared_error(true.flatten(), pred.flatten()))

                print("RMSE:", rmse)

                # Track best model
                if rmse < best_rmse:
                    best_rmse = rmse
                    best_params = {
                        "units": units,
                        "dropout": dropout,
                        "lr": lr,
                        "batch_size": batch
                    }
                    best_model = model

print("\n🎉 Best LSTM Hyperparameters Found:")
print(best_params)
print("Best RMSE:", best_rmse)

# -----------------------------
# 3. Save best model
# -----------------------------
best_model.save("models/tuned_lstm_week7.h5")
print("Saved tuned LSTM model → models/tuned_lstm_week7.h5")



Loaded existing player_synth_timeseries.csv with 185301 rows.
Using feature columns: ['market_value_in_eur', 'sentiment_score', 'injury_flag']
No sequences with SEQ_LEN=12. Reducing seq_len...
No sequences with SEQ_LEN=11. Reducing seq_len...
No sequences with SEQ_LEN=10. Reducing seq_len...
No sequences with SEQ_LEN=9. Reducing seq_len...
No sequences with SEQ_LEN=8. Reducing seq_len...
No sequences with SEQ_LEN=7. Reducing seq_len...
Built sequences: X shape = (20589, 6, 3), y shape = (20589, 3) (using seq_len=6)

Ready for LSTM tuning. Variables created:
SEQ_LEN: 6 OUT_STEPS: 3 nf: 3
Xtr_s shape: (16471, 6, 3) Xte_s shape: (4118, 6, 3)
ytr_s shape: (16471, 3, 1) yte_s shape: (4118, 3, 1)
🔍 Starting LSTM hyperparameter tuning...

Testing → units=64, dropout=0.1, lr=0.001, batch=64
RMSE: 0.01045688387013652

Testing → units=64, dropout=0.1, lr=0.001, batch=128
RMSE: 0.010675812749190454

Testing → units=64, dropout=0.1, lr=0.0005, batch=64
RMSE: 0.010881575147615025

Testing → units=6

RMSE: 0.013359386360795377

🎉 Best LSTM Hyperparameters Found:
{'units': 128, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}
Best RMSE: 0.009728594883870274
Saved tuned LSTM model → models/tuned_lstm_week7.h5


In [13]:
# === Evaluate tuned models & stacking ===
import joblib, numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def m_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred); rmse = mse**0.5
    return {"RMSE": float(rmse), "MAE": float(mean_absolute_error(y_true, y_pred)), "R2": float(r2_score(y_true, y_pred))}

results = {}

# Tuned XGBoost: we saved booster as xgb_tuned_week7.booster.json or sklearn wrapper pkl
import xgboost as xgb
if os.path.exists("xgb_tuned_week7.booster.json"):
    bst = xgb.Booster()
    bst.load_model("xgb_tuned_week7.booster.json")
    preds_xgb = bst.predict(xgb.DMatrix(X_test))
    results['XGBoost_Tuned'] = m_metrics(y_test, preds_xgb)
elif os.path.exists("xgb_tuned_week7.pkl"):
    xgb_wrapper = joblib.load("xgb_tuned_week7.pkl")
    preds_xgb = xgb_wrapper.predict(X_test)
    results['XGBoost_Tuned'] = m_metrics(y_test, preds_xgb)

# Tuned LightGBM
if os.path.exists("lgb_tuned_week7.txt"):
    gbm_tuned = lgb.Booster(model_file="lgb_tuned_week7.txt")
    preds_lgb = gbm_tuned.predict(X_test)
    results['LightGBM_Tuned'] = m_metrics(y_test, preds_lgb)
elif os.path.exists("lgb_tuned_week7.pkl"):
    gbm_wrap = joblib.load("lgb_tuned_week7.pkl")
    preds_lgb = gbm_wrap.predict(X_test)
    results['LightGBM_Tuned'] = m_metrics(y_test, preds_lgb)

# LSTM tuned (if any)
if os.path.exists("lstm_tuned_week7.h5"):
    from tensorflow.keras.models import load_model
    lstm = load_model("lstm_tuned_week7.h5")
    # requires proper X_test shape for LSTM; skip if not available
    try:
        preds_lstm = lstm.predict(Xte).flatten()
        results['LSTM_Tuned'] = m_metrics(yte.flatten(), preds_lstm)
    except:
        pass

# Stacked ensemble: rebuild meta features using tuned preds
meta_preds = None
if 'XGBoost_Tuned' in results and 'LightGBM_Tuned' in results:
    # create meta features
    meta_X_test = np.vstack([preds_xgb, preds_lgb]).T
    if os.path.exists("week6_meta_ridge.pkl"):
        meta = joblib.load("week6_meta_ridge.pkl")
        meta_pred = meta.predict(meta_X_test)
        results['Stacked_Tuned'] = m_metrics(y_test, meta_pred)

# Print & save
df_res = pd.DataFrame(results).T
print(df_res)
df_res.to_csv("week7_tuned_model_metrics.csv", index=True)
print("Saved week7_tuned_model_metrics.csv")


                    RMSE       MAE        R2
LightGBM_Tuned  0.003698  0.000224  0.999986
Saved week7_tuned_model_metrics.csv
